In [ ]:
# STAGE 6 — CLASSIFIER FINALIZATION
# 6.1 Environment Setup and Project Paths


from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import json
import glob

# Project directories
PROJECT_ROOT = "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final"

STAGE4_ROOT = os.path.join(
    PROJECT_ROOT, "Stage4_Segmentation"
)

STAGE5_ROOT = os.path.join(
    PROJECT_ROOT, "Stage5_Hybrid_Representation"
)

STAGE6_ROOT = os.path.join(
    PROJECT_ROOT, "Stage6_Classifier_Finalization"
)

# Create Stage 6 directory
os.makedirs(STAGE6_ROOT, exist_ok=True)

# Segmentation manifest (720 images)
SEGMENTATION_MANIFEST = os.path.join(
    STAGE4_ROOT, "segmentation_manifest.csv"
)

# Stage 5 predicted-mask manifest (720 images)
PREDICTED_MANIFEST = os.path.join(
    STAGE5_ROOT,
    "Stage5_1_Geometric_Features",
    "Results",
    "stage5_2_predicted_mask_inference_manifest_v1.csv"
)

print("Stage 6 environment initialized.")
print("Project root exists:", os.path.exists(PROJECT_ROOT))
print("Segmentation manifest exists:", os.path.exists(SEGMENTATION_MANIFEST))
print("Predicted manifest exists:", os.path.exists(PREDICTED_MANIFEST))
print("Stage 6 directory:", STAGE6_ROOT)

Mounted at /content/drive
Stage 6 environment initialized.
Project root exists: True
Segmentation manifest exists: True
Predicted manifest exists: True
Stage 6 directory: /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage6_Classifier_Finalization


In [ ]:
# Search for existing classifier checkpoints and split CSV files

print("CLASSIFIER CHECKPOINTS")

for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints"]]

    for file in files:
        if file.endswith((".pth", ".pt", ".ckpt")):
            print(os.path.join(root, file))


print("\nCLASSIFICATION SPLIT CSV FILES")

for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints"]]

    for file in files:
        if "split" in file.lower() and file.endswith(".csv"):
            print(os.path.join(root, file))

CLASSIFIER CHECKPOINTS
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage4_Segmentation/Experiment_4B_Joint_BCEDice/Checkpoints/efficientnet_b0_unet_best.pth
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage2A_Results/resnet50_baseline_best.pth
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage2A_Results/efficientnet_b0_best.pth

CLASSIFICATION SPLIT CSV FILES
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/betel_leaf_stratified_split.csv


In [ ]:
# STAGE 6.3 — Inspect Classification Split

CLASSIFICATION_SPLIT = os.path.join(
    PROJECT_ROOT,
    "betel_leaf_stratified_split.csv"
)

classification_df = pd.read_csv(CLASSIFICATION_SPLIT)

print("Classification split shape:", classification_df.shape)
print("\nColumns:")
print(classification_df.columns.tolist())

print("\nFirst five records:")
display(classification_df.head())

print("\nSplit distribution:")
print(classification_df["split"].value_counts())

print("\nClass distribution:")
print(pd.crosstab(
    classification_df["class"],
    classification_df["split"]
))

print("\nTotal images:", len(classification_df))

Classification split shape: (720, 4)

Columns:
['filename', 'path', 'class', 'split']

First five records:


,filename,path,class,split
0,Bacterial_Leaf_Spot_Disease(105).jpg,/content/drive/MyDrive/Projectwork/Betel_Leaf_...,bacterial,test
1,Bacterial_Leaf_Spot_Disease(106).jpg,/content/drive/MyDrive/Projectwork/Betel_Leaf_...,bacterial,test
2,Bacterial_Leaf_Spot_Disease(111).jpg,/content/drive/MyDrive/Projectwork/Betel_Leaf_...,bacterial,test
3,Bacterial_Leaf_Spot_Disease(114).jpg,/content/drive/MyDrive/Projectwork/Betel_Leaf_...,bacterial,test
4,Bacterial_Leaf_Spot_Disease(118).jpg,/content/drive/MyDrive/Projectwork/Betel_Leaf_...,bacterial,test



Split distribution:
split
train         504
test          108
validation    108
Name: count, dtype: int64

Class distribution:
split      test  train  validation
class                             
bacterial    37    171          37
fungal       34    160          34
healthy      37    173          37

Total images: 720


In [ ]:
# STAGE 6.4 — Dataset Compatibility and Classifier Location

# Load segmentation manifest
segmentation_df = pd.read_csv(SEGMENTATION_MANIFEST)

# Compare the two datasets
comparison = classification_df[
    ["filename", "class", "split"]
].merge(
    segmentation_df[["filename", "class", "split"]],
    on="filename",
    how="outer",
    suffixes=("_cls", "_seg"),
    indicator=True
)

print("DATASET COMPATIBILITY")
print("Classification images:", len(classification_df))
print("Segmentation images:", len(segmentation_df))
print("Filename overlap:", (comparison["_merge"] == "both").sum())
print("Missing from segmentation:", (comparison["_merge"] == "left_only").sum())
print("Missing from classification:", (comparison["_merge"] == "right_only").sum())

matched = comparison[comparison["_merge"] == "both"]

print("Class assignments match:",
      (matched["class_cls"] == matched["class_seg"]).all())

print("Split assignments match:",
      (matched["split_cls"] == matched["split_seg"]).all())


# Locate EfficientNet-B0 classifier files
print("\nCLASSIFIER FILE LOCATIONS")

search_root = "/content/drive/MyDrive"

for root, dirs, files in os.walk(search_root):
    dirs[:] = [d for d in dirs if d != ".ipynb_checkpoints"]

    for file in files:
        if file.lower() in [
            "efficientnetb0.pth",
            "efficientnetb0_history.csv"
        ]:
            print(os.path.join(root, file))

DATASET COMPATIBILITY
Classification images: 720
Segmentation images: 720
Filename overlap: 720
Missing from segmentation: 0
Missing from classification: 0
Class assignments match: True
Split assignments match: True

CLASSIFIER FILE LOCATIONS


In [ ]:
# Locate Stage2A_Results and EfficientNet-B0 files

search_root = "/content/drive/MyDrive/Projectwork"

for root, dirs, files in os.walk(search_root):
    if "Stage2A_Results" in root:
        print("\nFolder:", root)

        for file in files:
            if file.endswith((".pth", ".pt", ".csv", ".json")):
                print(" ", file)


Folder: /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage2A_Results
  resnet50_baseline_best.pth
  efficientnet_b0_best.pth
  efficientnet_b0_history.csv


In [ ]:
# STAGE 6.5 — Inspect EfficientNet-B0 Classifier

CLASSIFIER_DIR = os.path.join(PROJECT_ROOT, "Stage2A_Results")

CLASSIFIER_CHECKPOINT = os.path.join(
    CLASSIFIER_DIR, "efficientnet_b0_best.pth"
)

CLASSIFIER_HISTORY = os.path.join(
    CLASSIFIER_DIR, "efficientnet_b0_history.csv"
)

# Load training history
history_df = pd.read_csv(CLASSIFIER_HISTORY)

print("TRAINING HISTORY")
print("Shape:", history_df.shape)
print("Columns:", history_df.columns.tolist())

display(history_df.tail(10))

# Inspect checkpoint metadata
import torch

checkpoint = torch.load(
    CLASSIFIER_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\nCHECKPOINT INFORMATION")
print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if not isinstance(value, (dict, list, tuple, torch.Tensor)):
            print(f"{key}: {value}")

print("\nCheckpoint size (MB):",
      round(os.path.getsize(CLASSIFIER_CHECKPOINT) / (1024**2), 2))

TRAINING HISTORY
Shape: (15, 7)
Columns: ['epoch', 'train_loss', 'train_accuracy', 'val_loss', 'val_accuracy', 'val_macro_f1', 'epoch_time_sec']


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_f1,epoch_time_sec
5,6,0.108072,0.980159,0.138979,0.953704,0.954947,108.877743
6,7,0.115876,0.968254,0.118687,0.962963,0.963964,107.229950
7,8,0.067707,0.990079,0.109947,0.972222,0.972968,104.400474
8,9,0.074445,0.986111,0.101177,0.962963,0.963964,102.480910
9,10,0.044339,0.990079,0.099537,0.972222,0.972968,105.202518
10,11,0.033578,0.990079,0.105121,0.962963,0.963452,114.963387
11,12,0.055114,0.978175,0.095248,0.972222,0.972968,114.579370
12,13,0.030451,0.992063,0.094170,0.981481,0.981982,111.094873
13,14,0.024971,0.996032,0.095242,0.972222,0.972968,105.930413
14,15,0.014517,1.000000,0.095332,0.972222,0.972968,107.804986



CHECKPOINT INFORMATION
Checkpoint type: <class 'dict'>
Checkpoint keys: ['model_state_dict', 'classes', 'class_to_idx', 'best_epoch', 'best_val_macro_f1', 'seed', 'image_size', 'batch_size', 'learning_rate']
best_epoch: 13
best_val_macro_f1: 0.981981981981982
seed: 42
image_size: 224
batch_size: 32
learning_rate: 0.0001

Checkpoint size (MB): 15.59


In [ ]:
# STAGE 6.6 — Verify Classifier Architecture and Class Mapping

print("CLASS MAPPING")

print("Classes:", checkpoint["classes"])
print("Class-to-index:", checkpoint["class_to_idx"])

print("\nMODEL STATE DICTIONARY")

state_dict = checkpoint["model_state_dict"]

print("Number of parameter entries:", len(state_dict))
print("\nFirst 15 parameter names:")

for key in list(state_dict.keys())[:15]:
    print(key, state_dict[key].shape)

print("\nFinal classifier layer:")

for key, value in state_dict.items():
    if "classifier" in key:
        print(key, value.shape)

CLASS MAPPING
Classes: ['bacterial', 'fungal', 'healthy']
Class-to-index: {'bacterial': 0, 'fungal': 1, 'healthy': 2}

MODEL STATE DICTIONARY
Number of parameter entries: 360

First 15 parameter names:
features.0.0.weight torch.Size([32, 3, 3, 3])
features.0.1.weight torch.Size([32])
features.0.1.bias torch.Size([32])
features.0.1.running_mean torch.Size([32])
features.0.1.running_var torch.Size([32])
features.0.1.num_batches_tracked torch.Size([])
features.1.0.block.0.0.weight torch.Size([32, 1, 3, 3])
features.1.0.block.0.1.weight torch.Size([32])
features.1.0.block.0.1.bias torch.Size([32])
features.1.0.block.0.1.running_mean torch.Size([32])
features.1.0.block.0.1.running_var torch.Size([32])
features.1.0.block.0.1.num_batches_tracked torch.Size([])
features.1.0.block.1.fc1.weight torch.Size([8, 32, 1, 1])
features.1.0.block.1.fc1.bias torch.Size([8])
features.1.0.block.1.fc2.weight torch.Size([32, 8, 1, 1])

Final classifier layer:
classifier.1.weight torch.Size([3, 1280])
classif

In [ ]:
# STAGE 6.7 — Load EfficientNet-B0 Classifier

import torch
import torch.nn as nn
from torchvision import models

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Reconstruct architecture
classifier_model = models.efficientnet_b0(weights=None)

classifier_model.classifier[1] = nn.Linear(
    classifier_model.classifier[1].in_features,
    len(checkpoint["classes"])
)

# Load trained weights
classifier_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

classifier_model = classifier_model.to(device)
classifier_model.eval()

# Verify model
total_params = sum(
    p.numel() for p in classifier_model.parameters()
)

print("\nClassifier loaded successfully.")
print("Architecture: EfficientNet-B0")
print("Parameters:", f"{total_params:,}")
print("Input size:", checkpoint["image_size"])
print("Classes:", checkpoint["classes"])
print("Best epoch:", checkpoint["best_epoch"])
print("Best validation Macro-F1:", checkpoint["best_val_macro_f1"])

Device: cuda
GPU: Tesla T4

Classifier loaded successfully.
Architecture: EfficientNet-B0
Parameters: 4,011,391
Input size: 224
Classes: ['bacterial', 'fungal', 'healthy']
Best epoch: 13
Best validation Macro-F1: 0.981981981981982


In [ ]:
# STAGE 6.8 — Reproduce Original Evaluation Preprocessing

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGE_SIZE = 224
BATCH_SIZE = 32

# Exact evaluation transformation from Stage 2A notebook
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Dataset class following the original notebook
class BetelLeafDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")
        label = checkpoint["class_to_idx"][row["class"]]

        if self.transform:
            image = self.transform(image)

        return image, label


# Select held-out test images
test_df = classification_df[
    classification_df["split"] == "test"
].copy()

# Verify image paths
missing_paths = [
    p for p in test_df["path"]
    if not os.path.exists(p)
]

print("Test images:", len(test_df))
print("Missing image files:", len(missing_paths))

if missing_paths:
    print("Examples:", missing_paths[:5])
    raise FileNotFoundError("Some test images are missing.")

# Create test dataset and loader
test_dataset = BetelLeafDataset(
    test_df,
    transform=eval_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Verify one batch
images, labels = next(iter(test_loader))

print("\nTEST DATASET READY")
print("Number of test images:", len(test_dataset))
print("Number of batches:", len(test_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Class mapping:", checkpoint["class_to_idx"])

Test images: 108
Missing image files: 0

TEST DATASET READY
Number of test images: 108
Number of batches: 4
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Class mapping: {'bacterial': 0, 'fungal': 1, 'healthy': 2}


In [ ]:
# STAGE 6.9 — EFFICIENTNET-B0 HELD-OUT TEST EVALUATION

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

classifier_model.eval()

all_labels = []
all_predictions = []
all_probabilities = []
test_loss = 0.0
criterion = nn.CrossEntropyLoss()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = classifier_model(images)
        loss = criterion(outputs, labels)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(outputs, dim=1)

        test_loss += loss.item() * images.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())


# Convert results
all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)

classes = checkpoint["classes"]

# Calculate metrics
test_loss /= len(test_dataset)

test_accuracy = accuracy_score(
    all_labels, all_predictions
)

precision, recall, f1, _ = precision_recall_fscore_support(
    all_labels,
    all_predictions,
    labels=[0, 1, 2],
    average="macro",
    zero_division=0
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        all_labels,
        all_predictions,
        labels=[0, 1, 2],
        average="weighted",
        zero_division=0
    )
)

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1, 2]
)

# Display results
print("EFFICIENTNET-B0 — HELD-OUT TEST RESULTS")

print(f"Test images:          {len(test_dataset)}")
print(f"Test loss:            {test_loss:.4f}")
print(f"Test accuracy:        {test_accuracy:.4f}")
print(f"Test Macro Precision: {precision:.4f}")
print(f"Test Macro Recall:    {recall:.4f}")
print(f"Test Macro-F1:        {f1:.4f}")
print(f"Weighted Precision:   {weighted_precision:.4f}")
print(f"Weighted Recall:      {weighted_recall:.4f}")
print(f"Weighted F1:          {weighted_f1:.4f}")

print("\nCLASSIFICATION REPORT")

print(classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2],
    target_names=classes,
    digits=4,
    zero_division=0
))

print("CONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted")
print("Class order:", classes)
print(cm)

EFFICIENTNET-B0 — HELD-OUT TEST RESULTS
Test images:          108
Test loss:            0.0726
Test accuracy:        0.9815
Test Macro Precision: 0.9822
Test Macro Recall:    0.9812
Test Macro-F1:        0.9816
Weighted Precision:   0.9817
Weighted Recall:      0.9815
Weighted F1:          0.9815

CLASSIFICATION REPORT
              precision    recall  f1-score   support

   bacterial     0.9730    0.9730    0.9730        37
      fungal     1.0000    0.9706    0.9851        34
     healthy     0.9737    1.0000    0.9867        37

    accuracy                         0.9815       108
   macro avg     0.9822    0.9812    0.9816       108
weighted avg     0.9817    0.9815    0.9815       108

CONFUSION MATRIX
Rows = Actual | Columns = Predicted
Class order: ['bacterial', 'fungal', 'healthy']
[[36  0  1]
 [ 1 33  0]
 [ 0  0 37]]


In [ ]:
# STAGE 6.10 — SAVE CLASSIFIER TEST RESULTS

from sklearn.metrics import classification_report

# Results directory
RESULTS_DIR = os.path.join(STAGE6_ROOT, "Results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# 1. Overall metrics
metrics = {
    "model": "EfficientNet-B0",
    "checkpoint": "efficientnet_b0_best.pth",
    "best_epoch": int(checkpoint["best_epoch"]),
    "best_validation_macro_f1": float(checkpoint["best_val_macro_f1"]),
    "test_images": len(test_dataset),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_macro_precision": float(precision),
    "test_macro_recall": float(recall),
    "test_macro_f1": float(f1),
    "test_weighted_precision": float(weighted_precision),
    "test_weighted_recall": float(weighted_recall),
    "test_weighted_f1": float(weighted_f1),
    "confusion_matrix": cm.tolist(),
    "classes": classes
}

metrics_path = os.path.join(
    RESULTS_DIR,
    "stage6_classifier_test_metrics_v1.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)


# 2. Class-wise classification report
report = classification_report(
    all_labels,
    all_predictions,
    labels=[0, 1, 2],
    target_names=classes,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_path = os.path.join(
    RESULTS_DIR,
    "stage6_classifier_classification_report_v1.csv"
)

report_df.to_csv(report_path)


# 3. Individual predictions
predictions_df = test_df[["filename", "class", "split"]].copy()

predictions_df["actual_label"] = [
    classes[i] for i in all_labels
]

predictions_df["predicted_label"] = [
    classes[i] for i in all_predictions
]

predictions_df["correct"] = (
    all_labels == all_predictions
)

for i, class_name in enumerate(classes):
    predictions_df[f"probability_{class_name}"] = (
        all_probabilities[:, i]
    )

predictions_path = os.path.join(
    RESULTS_DIR,
    "stage6_classifier_test_predictions_v1.csv"
)

predictions_df.to_csv(predictions_path, index=False)


print("Stage 6 test evaluation saved successfully.")
print("\nFiles:")
print(metrics_path)
print(report_path)
print(predictions_path)

Stage 6 test evaluation saved successfully.

Files:
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage6_Classifier_Finalization/Results/stage6_classifier_test_metrics_v1.json
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage6_Classifier_Finalization/Results/stage6_classifier_classification_report_v1.csv
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage6_Classifier_Finalization/Results/stage6_classifier_test_predictions_v1.csv
